## Example 1: Basic DNS Queries and Record Types

In [ ]:
import dns.resolver
import dns.query
import dns.name
import dns.rdatatype
import dns

print("Example 1: Basic DNS Queries and Record Types")

try:
    print("✓ Basic DNS queries and record types:")
    
    # 1. Package information
    print("  1. Package information:")
    print(f"    DNS package: {dns.__name__}")
    print(f"    DNS version: {getattr(dns, '__version__', 'unknown')}")
    
    # Show available modules
    dns_modules = [attr for attr in dir(dns) if not attr.startswith('_') and hasattr(getattr(dns, attr), '__name__')]
    print(f"    Available modules: {dns_modules[:8]}...")
    
    # 2. Basic A record queries
    print("  2. Basic A record queries:")
    
    # Query well-known domains
    test_domains = ['google.com', 'cloudflare.com', 'github.com']
    
    for domain in test_domains:
        try:
            # Perform A record lookup
            result = dns.resolver.resolve(domain, 'A')
            ip_addresses = [str(rdata) for rdata in result]
            
            print(f"    {domain} A records: {ip_addresses[:2]}...")  # Show first 2 IPs
            print(f"      TTL: {result.ttl} seconds")
            print(f"      Total records: {len(result)}")
            
        except Exception as e:
            print(f"    {domain} A record failed: {type(e).__name__}")
            raise
        
        # Small delay to be respectful to DNS servers
        import time
        time.sleep(0.1)
    
    # 3. Multiple record types
    print("  3. Multiple record types:")
    
    test_domain = 'google.com'
    record_types = ['A', 'AAAA', 'MX', 'TXT', 'NS']
    
    for record_type in record_types:
        try:
            result = dns.resolver.resolve(test_domain, record_type)
            records = [str(rdata) for rdata in result]
            
            print(f"    {test_domain} {record_type} records: {len(records)} found")
            if records:
                print(f"      Sample: {records[0][:50]}...")
                
        except dns.resolver.NoAnswer:
            print(f"    {test_domain} {record_type} records: No answer")
        except Exception as e:
            print(f"    {test_domain} {record_type} records: {type(e).__name__}")
            raise
        
        time.sleep(0.1)
    
    # 4. DNS record analysis
    print("  4. DNS record analysis:")
    
    # Analyze MX records for email domains
    email_domains = ['gmail.com', 'outlook.com']
    
    for domain in email_domains:
        try:
            mx_result = dns.resolver.resolve(domain, 'MX')
            mx_records = [(rdata.preference, str(rdata.exchange)) for rdata in mx_result]
            mx_records.sort(key=lambda x: x[0])  # Sort by preference
            
            print(f"    {domain} MX records (sorted by preference):")
            for preference, exchange in mx_records[:3]:  # Show top 3
                print(f"      Priority {preference}: {exchange}")
                
        except Exception as e:
            print(f"    {domain} MX analysis failed: {type(e).__name__}")
            raise
        
        time.sleep(0.1)
    
    # 5. DNS name manipulation
    print("  5. DNS name manipulation:")
    
    # Work with DNS names
    test_names = ['www.example.com', 'mail.google.com', 'api.github.com']
    
    for name_str in test_names:
        try:
            dns_name = dns.name.from_text(name_str)
            
            print(f"    DNS name: {name_str}")
            print(f"      Absolute: {dns_name}")
            print(f"      Labels: {len(dns_name.labels)}")
            print(f"      Parent: {dns_name.parent()}")
            
            # Check if it's a subdomain. Labels are bytes and an absolute name ends with the empty root
            # label b'', so the registered domain is the last three labels (b'example', b'com', b'').
            if len(dns_name.labels) > 3:
                root_domain = dns.name.Name(dns_name.labels[-3:])
                print(f"      Root domain: {root_domain}")
                assert root_domain.is_superdomain(dns_name), (root_domain, dns_name)
            
        except Exception as e:
            print(f"    {name_str} name manipulation failed: {type(e).__name__}")
            raise
    
    # 6. Record type information
    print("  6. Record type information:")
    
    # Show common record types
    common_types = [
        ('A', dns.rdatatype.A, 'IPv4 address'),
        ('AAAA', dns.rdatatype.AAAA, 'IPv6 address'), 
        ('MX', dns.rdatatype.MX, 'Mail exchange'),
        ('NS', dns.rdatatype.NS, 'Name server'),
        ('CNAME', dns.rdatatype.CNAME, 'Canonical name'),
        ('TXT', dns.rdatatype.TXT, 'Text record')
    ]
    
    print(f"    Common DNS record types:")
    for type_name, type_code, description in common_types:
        print(f"      {type_name} ({type_code}): {description}")
    
    # Test record type conversion
    print(f"    Record type examples:")
    print(f"      'A' -> {dns.rdatatype.from_text('A')}")
    print(f"      {dns.rdatatype.A} -> {dns.rdatatype.to_text(dns.rdatatype.A)}")
    
    print("✓ Basic DNS queries and record types completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: dnspython provides comprehensive DNS functionality")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dnspython example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced DNS Resolution and Server Configuration

In [ ]:
import dns.resolver
import dns.query
import dns.message
import dns.flags
import time

print("Example 2: Advanced DNS Resolution and Server Configuration")

try:
    print("✓ Advanced DNS resolution and server configuration:")
    
    # 1. Custom DNS resolver configuration
    print("  1. Custom DNS resolver configuration:")
    
    # Create custom resolver
    custom_resolver = dns.resolver.Resolver()
    
    # Configure different DNS servers
    dns_servers = [
        ('Google DNS', ['8.8.8.8', '8.8.4.4']),
        ('Cloudflare DNS', ['1.1.1.1', '1.0.0.1']),
        ('Quad9 DNS', ['9.9.9.9', '149.112.112.112'])
    ]
    
    print(f"    Default resolver nameservers: {dns.resolver.default_resolver.nameservers}")
    
    # Test different DNS servers
    test_domain = 'example.com'
    
    for server_name, nameservers in dns_servers[:2]:  # Test first 2 to avoid excessive queries
        try:
            custom_resolver.nameservers = nameservers
            custom_resolver.timeout = 2.0
            custom_resolver.lifetime = 5.0
            
            start_time = time.time()
            result = custom_resolver.resolve(test_domain, 'A')
            end_time = time.time()
            
            query_time = (end_time - start_time) * 1000
            ip_addresses = [str(rdata) for rdata in result]
            
            print(f"    {server_name} ({nameservers[0]}):")
            print(f"      Query time: {query_time:.1f}ms")
            print(f"      Result: {ip_addresses[0]}")
            print(f"      TTL: {result.ttl}s")
            
        except Exception as e:
            print(f"    {server_name} failed: {type(e).__name__}")
            raise
        
        time.sleep(0.2)
    
    # 2. DNS query flags and options
    print("  2. DNS query flags and options:")
    
    # Create DNS message with specific flags
    query_msg = dns.message.make_query('google.com', dns.rdatatype.A)
    
    # Set DNS flags
    query_msg.flags |= dns.flags.RD  # Recursion Desired
    query_msg.flags |= dns.flags.AD  # Authentic Data
    
    print(f"    Query message flags:")
    flag_names = []
    if query_msg.flags & dns.flags.QR:
        flag_names.append('QR')
    if query_msg.flags & dns.flags.AA:
        flag_names.append('AA') 
    if query_msg.flags & dns.flags.RD:
        flag_names.append('RD')
    if query_msg.flags & dns.flags.RA:
        flag_names.append('RA')
    if query_msg.flags & dns.flags.AD:
        flag_names.append('AD')
    
    print(f"      Active flags: {flag_names}")
    print(f"      Question count: {len(query_msg.question)}")
    print(f"      Question: {query_msg.question[0]}")
    
    # 3. Direct DNS server queries
    print("  3. Direct DNS server queries:")
    
    # Query specific DNS server directly
    try:
        dns_server = '8.8.8.8'
        
        # Send UDP query
        response = dns.query.udp(query_msg, dns_server, timeout=3)
        
        print(f"    Direct query to {dns_server}:")
        print(f"      Response ID: {response.id}")
        print(f"      Answer count: {len(response.answer)}")
        print(f"      Authority count: {len(response.authority)}")
        print(f"      Additional count: {len(response.additional)}")
        
        # Parse answers
        if response.answer:
            for answer_section in response.answer:
                print(f"      Answer: {answer_section.name} {answer_section.ttl} {answer_section.rdclass} {answer_section.rdtype}")
                for rdata in answer_section:
                    print(f"        -> {rdata}")
        
    except Exception as e:
        print(f"    Direct query failed: {type(e).__name__}")
        raise
    
    # 4. DNS caching behavior
    print("  4. DNS caching behavior:")
    
    # Test caching with multiple queries
    cache_test_domain = 'github.com'
    
    # First query (cache miss)
    start_time = time.time()
    result1 = dns.resolver.resolve(cache_test_domain, 'A')
    first_query_time = (time.time() - start_time) * 1000
    
    time.sleep(0.1)
    
    # Second query (potential cache hit)
    start_time = time.time()
    result2 = dns.resolver.resolve(cache_test_domain, 'A')
    second_query_time = (time.time() - start_time) * 1000
    
    print(f"    Cache test for {cache_test_domain}:")
    print(f"      First query: {first_query_time:.1f}ms")
    print(f"      Second query: {second_query_time:.1f}ms")
    print(f"      Results match: {str(result1[0]) == str(result2[0])}")
    print(f"      TTL difference: {abs(result1.ttl - result2.ttl)} seconds")
    
    # 5. Reverse DNS lookups
    print("  5. Reverse DNS lookups:")
    
    # Perform reverse DNS on well-known IPs
    well_known_ips = ['8.8.8.8', '1.1.1.1', '208.67.222.222']
    
    for ip_addr in well_known_ips:
        try:
            # Create reverse lookup address
            reverse_name = dns.reversename.from_address(ip_addr)
            
            # Perform PTR query
            result = dns.resolver.resolve(reverse_name, 'PTR')
            hostnames = [str(rdata) for rdata in result]
            
            print(f"    Reverse DNS for {ip_addr}:")
            print(f"      Reverse name: {reverse_name}")
            print(f"      Hostnames: {hostnames}")
            
        except Exception as e:
            print(f"    Reverse DNS for {ip_addr} failed: {type(e).__name__}")
            raise
        
        time.sleep(0.1)
    
    # 6. DNS query statistics and performance
    print("  6. DNS query performance analysis:")
    
    # Perform multiple queries and analyze performance
    performance_domains = ['google.com', 'amazon.com', 'microsoft.com']
    query_times = []
    
    for domain in performance_domains:
        try:
            start_time = time.time()
            result = dns.resolver.resolve(domain, 'A')
            end_time = time.time()
            
            query_time = (end_time - start_time) * 1000
            query_times.append(query_time)
            
            print(f"    {domain}: {query_time:.1f}ms (TTL: {result.ttl}s)")
            
        except Exception as e:
            print(f"    {domain} performance test failed: {type(e).__name__}")
            raise
        
        time.sleep(0.1)
    
    if query_times:
        avg_time = sum(query_times) / len(query_times)
        min_time = min(query_times)
        max_time = max(query_times)
        
        print(f"    Performance summary:")
        print(f"      Average: {avg_time:.1f}ms")
        print(f"      Min: {min_time:.1f}ms")
        print(f"      Max: {max_time:.1f}ms")
        print(f"      Queries tested: {len(query_times)}")
    
    print("✓ Advanced DNS resolution and server configuration completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: dnspython supports advanced DNS resolution features")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dnspython example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: DNS Zone Management and Record Creation

In [ ]:
import dns.zone
import dns.rdataset
import dns.rdataclass
import dns.rdata
import dns.exception
import dns.flags
import dns.message
import dns.name
import dns.rdatatype
import dns.rrset
import io
import textwrap

print("Example 3: DNS Zone Management and Record Creation")

try:
    print("✓ DNS zone management and record creation:")
    
    # 1. Create DNS zone from text
    print("  1. Create DNS zone from text:")
    
    # Define a sample zone file content
    zone_text = """
    example.local.  IN  SOA  ns1.example.local. admin.example.local. (
                        2023010101  ; Serial
                        3600        ; Refresh
                        1800        ; Retry
                        604800      ; Expire
                        86400       ; Minimum TTL
                    )
    example.local.      IN  NS   ns1.example.local.
    example.local.      IN  NS   ns2.example.local.
    example.local.      IN  A    192.168.1.10
    www.example.local.  IN  A    192.168.1.11
    mail.example.local. IN  A    192.168.1.12
    example.local.      IN  MX   10 mail.example.local.
    ftp.example.local.  IN  CNAME www.example.local.
    """
    
    try:
        # Parse zone from text
        zone_name = dns.name.from_text('example.local.')
        # Master-file syntax: a line that starts with whitespace inherits the PREVIOUS owner name, so the
        # indented text must be dedented or every owner is misread as a type ("unknown rdatatype").
        zone = dns.zone.from_text(textwrap.dedent(zone_text), zone_name)
        
        print(f"    Zone created for: {zone.origin}")
        print(f"    Zone class: {dns.rdataclass.to_text(zone.rdclass)}")
        
        # List all nodes in the zone
        node_count = 0
        record_count = 0
        
        for name, node in zone.nodes.items():
            node_count += 1
            
            # Get full name
            full_name = name.derelativize(zone.origin)
            print(f"    Node: {full_name}")
            
            # List record sets for this node
            for rdataset in node.rdatasets:
                record_count += len(rdataset)
                rdtype_text = dns.rdatatype.to_text(rdataset.rdtype)
                
                print(f"      {rdtype_text} records: {len(rdataset)}")
                for rdata in rdataset:
                    print(f"        -> {rdata}")
        
        print(f"    Zone statistics: {node_count} nodes, {record_count} records")
        assert node_count == 4 and record_count == 8, (node_count, record_count)  # @: SOA+2 NS+A+MX; www, mail: A; ftp: CNAME
        
    except Exception as e:
        print(f"    Zone creation failed: {type(e).__name__}: {e}")
        raise
    
    # 2. Create individual DNS records
    print("  2. Create individual DNS records:")
    
    try:
        # Create different types of DNS records
        record_examples = [
            ('A', '192.168.1.100'),
            ('AAAA', '2001:db8::1'),
            ('CNAME', 'target.example.com.'),
            ('MX', '10 mail.example.com.'),
            ('TXT', '"This is a text record"')
        ]
        
        for record_type, record_data in record_examples:
            try:
                # Create record data
                rdtype = dns.rdatatype.from_text(record_type)
                rdclass = dns.rdataclass.IN
                
                rdata = dns.rdata.from_text(rdclass, rdtype, record_data)
                
                print(f"    {record_type} record created:")
                print(f"      Type: {dns.rdatatype.to_text(rdtype)} ({rdtype})")
                print(f"      Class: {dns.rdataclass.to_text(rdclass)}")
                print(f"      Data: {rdata}")
                print(f"      Wire format length: {len(rdata.to_wire())} bytes")
                
            except Exception as e:
                print(f"    {record_type} record creation failed: {type(e).__name__}")
                raise
    
    except Exception as e:
        print(f"    Record creation failed: {type(e).__name__}")
        raise
    
    # 3. Zone file operations
    print("  3. Zone file operations:")
    
    try:
        if 'zone' in locals():
            # Convert zone back to text
            zone_output = io.StringIO()
            zone.to_file(zone_output)
            zone_text_output = zone_output.getvalue()
            
            print(f"    Zone file export:")
            lines = zone_text_output.strip().split('\n')
            print(f"      Total lines: {len(lines)}")
            print(f"      Sample lines:")
            
            for i, line in enumerate(lines[:5]):  # Show first 5 lines
                print(f"        {i+1}: {line.strip()}")
            
            if len(lines) > 5:
                print(f"        ... and {len(lines) - 5} more lines")
            
            # Verify zone integrity
            soa_records = []
            ns_records = []
            a_records = []
            
            for name, node in zone.nodes.items():
                for rdataset in node.rdatasets:
                    rdtype = rdataset.rdtype
                    if rdtype == dns.rdatatype.SOA:
                        soa_records.extend(rdataset)
                    elif rdtype == dns.rdatatype.NS:
                        ns_records.extend(rdataset)
                    elif rdtype == dns.rdatatype.A:
                        a_records.extend(rdataset)
            
            print(f"    Zone validation:")
            print(f"      SOA records: {len(soa_records)}")
            print(f"      NS records: {len(ns_records)}")
            print(f"      A records: {len(a_records)}")
            
            # Check SOA record details
            if soa_records:
                soa = soa_records[0]
                print(f"      SOA details:")
                print(f"        MNAME: {soa.mname}")
                print(f"        RNAME: {soa.rname}")
                print(f"        Serial: {soa.serial}")
                print(f"        Refresh: {soa.refresh}s")
        
    except Exception as e:
        print(f"    Zone file operations failed: {type(e).__name__}")
        raise
    
    # 4. Record set manipulation
    print("  4. Record set manipulation:")
    
    try:
        # Create and manipulate record sets
        rdataset_a = dns.rdataset.Rdataset(dns.rdataclass.IN, dns.rdatatype.A)
        
        # Add multiple A records
        ip_addresses = ['192.168.1.20', '192.168.1.21', '192.168.1.22']
        
        for ip in ip_addresses:
            rdata = dns.rdata.from_text(dns.rdataclass.IN, dns.rdatatype.A, ip)
            rdataset_a.add(rdata)
        
        print(f"    Record set created:")
        print(f"      Type: {dns.rdatatype.to_text(rdataset_a.rdtype)}")
        print(f"      Class: {dns.rdataclass.to_text(rdataset_a.rdclass)}")
        print(f"      Record count: {len(rdataset_a)}")
        
        # List all records in the set
        print(f"      Records:")
        for i, rdata in enumerate(rdataset_a):
            print(f"        {i+1}: {rdata}")
        
        # Set TTL
        rdataset_a.ttl = 300
        print(f"      TTL set to: {rdataset_a.ttl} seconds")
        
    except Exception as e:
        print(f"    Record set manipulation failed: {type(e).__name__}")
        raise
    
    # 5. DNS message construction
    print("  5. DNS message construction:")
    
    try:
        # Create a complete DNS response message
        response_msg = dns.message.Message()
        
        # Set message properties
        response_msg.id = 12345
        response_msg.flags |= dns.flags.QR  # Query Response
        response_msg.flags |= dns.flags.AA  # Authoritative Answer
        response_msg.flags |= dns.flags.RD  # Recursion Desired
        response_msg.flags |= dns.flags.RA  # Recursion Available
        
        # Create question section
        # Owner names are ABSOLUTE (trailing dot): rrset.from_text() without an origin keeps a dotless name
        # relative, and to_wire() then raises NeedAbsoluteNameOrOrigin.
        question = dns.rrset.from_text('test.example.com.', 300, dns.rdataclass.IN, dns.rdatatype.A)
        response_msg.question.append(dns.rrset.RRset(question.name, question.rdclass, question.rdtype))
        
        # Create answer section
        answer_rrset = dns.rrset.from_text('test.example.com.', 300, dns.rdataclass.IN, dns.rdatatype.A, '192.168.1.50')
        response_msg.answer.append(answer_rrset)
        
        # Create authority section (NS record)
        authority_rrset = dns.rrset.from_text('example.com.', 3600, dns.rdataclass.IN, dns.rdatatype.NS, 'ns1.example.com.')
        response_msg.authority.append(authority_rrset)
        
        print(f"    DNS message constructed:")
        print(f"      Message ID: {response_msg.id}")
        print(f"      Flags: {response_msg.flags}")
        print(f"      Questions: {len(response_msg.question)}")
        print(f"      Answers: {len(response_msg.answer)}")
        print(f"      Authority: {len(response_msg.authority)}")
        print(f"      Additional: {len(response_msg.additional)}")
        
        # Check message size
        wire_data = response_msg.to_wire()
        print(f"      Wire format size: {len(wire_data)} bytes")
        
        # Verify message can be parsed back
        parsed_msg = dns.message.from_wire(wire_data)
        print(f"      Message parsing: {'Success' if parsed_msg.id == response_msg.id else 'Failed'}")
        assert parsed_msg.id == response_msg.id and len(parsed_msg.answer) == 1, parsed_msg
        
    except Exception as e:
        print(f"    DNS message construction failed: {type(e).__name__}")
        raise
    
    print("✓ DNS zone management and record creation completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: dnspython provides comprehensive DNS zone and record management")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dnspython example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: DNS Security Extensions (DNSSEC) and Validation

In [ ]:
import dns.dnssec
import dns.flags
import dns.rrset
import dns.rdatatype
import dns.resolver
import dns.exception
import time
import hashlib

print("Example 4: DNS Security Extensions (DNSSEC) and Validation")

try:
    print("✓ DNS security extensions and validation:")
    
    # 1. DNSSEC-aware DNS queries
    print("  1. DNSSEC-aware DNS queries:")
    
    # Create DNSSEC-aware resolver
    dnssec_resolver = dns.resolver.Resolver()
    # DO (DNSSEC OK) is an EDNS flag, so it goes in use_edns(). set_flags() sets the HEADER flags, where the same
    # bit (0x8000) means QR: the query would go out marked as a response and resolvers would drop it.
    dnssec_resolver.use_edns(0, dns.flags.DO, 1232)
    assert dnssec_resolver.ednsflags & dns.flags.DO and not (dnssec_resolver.flags or 0) & dns.flags.QR
    
    # Test DNSSEC-signed domains
    dnssec_domains = ['cloudflare.com', 'ietf.org', 'verisign.com']
    
    for domain in dnssec_domains[:2]:  # Test first 2 to avoid excessive queries
        try:
            print(f"    DNSSEC query for {domain}:")
            
            # Query for A record with DNSSEC
            start_time = time.time()
            result = dnssec_resolver.resolve(domain, 'A')
            query_time = (time.time() - start_time) * 1000
            
            print(f"      Query time: {query_time:.1f}ms")
            print(f"      A records: {len(result)}")
            print(f"      TTL: {result.ttl}s")
            
            # Signatures come back in the same answer when the resolver passes them on (a direct query for
            # type RRSIG gets NoAnswer or SERVFAIL from many resolvers, so it is not sent)
            rrsigs = [rrset for rrset in result.response.answer if rrset.rdtype == dns.rdatatype.RRSIG]
            print(f"      RRSIG records in the answer: {sum(len(rrset) for rrset in rrsigs)}")
            
            try:
                dnskey_result = dnssec_resolver.resolve(domain, 'DNSKEY')
                print(f"      DNSKEY records found: {len(dnskey_result)}")
            except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN):
                print(f"      DNSKEY records: Not found")
            except (dns.resolver.NoNameservers, dns.exception.Timeout):
                # some stub resolvers answer SERVFAIL (or nothing) for DNSKEY
                print(f"      DNSKEY records: not available from this resolver")
                
        except Exception as e:
            print(f"    DNSSEC query for {domain} failed: {type(e).__name__}")
            raise
        
        time.sleep(0.2)
    
    # 2. DNS record validation and integrity
    print("  2. DNS record validation and integrity:")
    
    # Create sample records for validation
    validation_domain = 'test.example.com'
    
    # Create A record
    a_rrset = dns.rrset.from_text(validation_domain, 300, dns.rdataclass.IN, dns.rdatatype.A, '192.168.1.100')
    
    # Create AAAA record
    aaaa_rrset = dns.rrset.from_text(validation_domain, 300, dns.rdataclass.IN, dns.rdatatype.AAAA, '2001:db8::1')
    
    print(f"    Record validation for {validation_domain}:")
    
    # Validate A record structure
    print(f"      A record validation:")
    print(f"        Name: {a_rrset.name}")
    print(f"        TTL: {a_rrset.ttl}")
    print(f"        Type: {dns.rdatatype.to_text(a_rrset.rdtype)}")
    print(f"        Class: {dns.rdataclass.to_text(a_rrset.rdclass)}")
    print(f"        Data count: {len(a_rrset)}")
    
    for i, rdata in enumerate(a_rrset):
        print(f"        Record {i+1}: {rdata}")
        
        # Validate IP address format
        ip_str = str(rdata)
        ip_parts = ip_str.split('.')
        if len(ip_parts) == 4 and all(0 <= int(part) <= 255 for part in ip_parts if part.isdigit()):
            print(f"          IP validation: Valid IPv4 address")
        else:
            print(f"          IP validation: Invalid IPv4 address")
    
    # Validate AAAA record
    print(f"      AAAA record validation:")
    for i, rdata in enumerate(aaaa_rrset):
        ipv6_str = str(rdata)
        print(f"        Record {i+1}: {rdata}")
        print(f"          IPv6 validation: {'Valid' if ':' in ipv6_str else 'Invalid'} format")
    
    assert (a_rrset.ttl, len(a_rrset), [str(r) for r in a_rrset]) == (300, 1, ["192.168.1.100"]), a_rrset
    assert [str(r) for r in aaaa_rrset] == ["2001:db8::1"], aaaa_rrset

    # 3. DNS name validation and normalization
    print("  3. DNS name validation and normalization:")
    
    test_names = [
        'example.com',
        'www.example.com.',  # Already absolute
        'SUB.DOMAIN.EXAMPLE.COM',  # Mixed case
        'xn--nxasmq6b.example.com',  # IDN (internationalized domain)
        'very-long-subdomain-name.example.org'
    ]
    
    for name_str in test_names:
        try:
            print(f"    Name: {name_str}")
            
            # Parse DNS name
            dns_name = dns.name.from_text(name_str)
            
            # Check if absolute
            is_absolute = dns_name.is_absolute()
            print(f"      Absolute: {is_absolute}")
            
            # Convert to absolute if needed
            if not is_absolute:
                abs_name = dns_name.concatenate(dns.name.root)
                print(f"      Absolute form: {abs_name}")
            else:
                abs_name = dns_name
            
            # Canonical form (lowercase)
            canonical = abs_name.to_text().lower()
            print(f"      Canonical: {canonical}")
            
            # Label analysis
            print(f"      Labels: {len(dns_name.labels)}")
            print(f"      Parent domain: {dns_name.parent()}")
            
            # Name length validation
            wire_name = dns_name.to_wire()
            print(f"      Wire format length: {len(wire_name)} bytes")
            
            if len(wire_name) > 255:
                print(f"      Warning: Name exceeds maximum length (255 bytes)")
            else:
                print(f"      Length validation: OK")
                
        except Exception as e:
            print(f"    Name validation for {name_str} failed: {type(e).__name__}")
            raise
    
    # 4. DNS message integrity checks
    print("  4. DNS message integrity checks:")
    
    try:
        # Create test DNS query message
        test_query = dns.message.make_query('security-test.example.com', dns.rdatatype.A)
        
        # Add EDNS for larger message support
        test_query.use_edns(payload=4096)
        
        print(f"    DNS message integrity:")
        print(f"      Message ID: {test_query.id}")
        print(f"      Opcode: {test_query.opcode()}")
        print(f"      EDNS enabled: {test_query.edns >= 0}")
        
        # Convert to wire format
        wire_data = test_query.to_wire()
        print(f"      Wire format size: {len(wire_data)} bytes")
        
        # Create message hash for integrity
        message_hash = hashlib.sha256(wire_data).hexdigest()
        print(f"      Message hash (SHA-256): {message_hash[:16]}...")
        
        # Parse message back from wire format
        parsed_msg = dns.message.from_wire(wire_data)
        
        # Verify integrity
        integrity_checks = [
            ('Message ID', test_query.id == parsed_msg.id),
            ('Question count', len(test_query.question) == len(parsed_msg.question)),
            ('EDNS support', test_query.edns == parsed_msg.edns),
            ('Flags', test_query.flags == parsed_msg.flags)
        ]
        
        print(f"      Integrity checks:")
        for check_name, check_result in integrity_checks:
            status = "✓" if check_result else "✗"
            print(f"        {status} {check_name}: {'Pass' if check_result else 'Fail'}")
        
        # Verify wire format reproducibility
        reparsed_wire = parsed_msg.to_wire()
        wire_match = wire_data == reparsed_wire
        print(f"      Wire format reproducibility: {'✓ Pass' if wire_match else '✗ Fail'}")
        
    except Exception as e:
        print(f"    Message integrity checks failed: {type(e).__name__}")
        raise
    
    # 5. DNS query security analysis
    print("  5. DNS query security analysis:")
    
    try:
        # Analyze potential security issues in DNS queries
        security_test_cases = [
            ('Normal query', 'example.com', 'A'),
            ('Long domain', 'a' * 60 + '.example.com', 'A'),
            ('Deep subdomain', '.'.join(['sub'] * 10) + '.example.com', 'A'),
            ('Multiple types', 'example.com', 'ANY'),  # ANY queries can be abused
        ]
        
        for test_name, domain, record_type in security_test_cases:
            try:
                print(f"    {test_name}:")
                
                # Create query
                query_msg = dns.message.make_query(domain, record_type)
                wire_size = len(query_msg.to_wire())
                
                print(f"      Domain: {domain}")
                print(f"      Type: {record_type}")
                print(f"      Query size: {wire_size} bytes")
                
                # Security analysis
                security_notes = []
                
                if len(domain) > 253:  # RFC limit
                    security_notes.append("Domain name exceeds RFC length limit")
                
                if domain.count('.') > 127:  # Label limit
                    security_notes.append("Excessive subdomain levels")
                
                if record_type == 'ANY':
                    security_notes.append("ANY queries can be used for amplification attacks")
                
                if wire_size > 512:  # Traditional UDP limit
                    security_notes.append("Query may require TCP or EDNS")
                
                if security_notes:
                    print(f"      Security notes:")
                    for note in security_notes:
                        print(f"        ⚠ {note}")
                else:
                    print(f"      Security analysis: No issues detected")
                
            except Exception as e:
                print(f"    {test_name} analysis failed: {type(e).__name__}")
                raise
        
        # DNS amplification analysis
        print(f"    DNS amplification analysis:")
        
        # Simulate potential amplification factors
        query_types_amp = [
            ('A', 'Small response'),
            ('MX', 'Medium response'),
            ('TXT', 'Variable response'),
            ('DNSKEY', 'Large response (DNSSEC)'),
        ]
        
        base_query_size = 28  # Approximate minimum DNS query size
        
        for record_type, description in query_types_amp:
            # Estimate response sizes (these are rough estimates)
            estimated_sizes = {
                'A': 32,      # Small, single IP
                'MX': 64,     # Medium, mail servers
                'TXT': 128,   # Variable, can be large
                'DNSKEY': 512 # Large, public keys
            }
            
            estimated_response = estimated_sizes.get(record_type, 64)
            amplification_factor = estimated_response / base_query_size
            
            print(f"      {record_type} ({description}):")
            print(f"        Est. response size: {estimated_response} bytes")
            print(f"        Amplification factor: {amplification_factor:.1f}x")
    
    except Exception as e:
        print(f"    Security analysis failed: {type(e).__name__}")
        raise
    
    print("✓ DNS security extensions and validation completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dnspython example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: DNS Utilities and Network Diagnostics

In [ ]:
import dns.resolver
import dns.reversename
import dns.inet
import dns.ipv4
import dns.ipv6
import time
import socket

print("Example 5: DNS Utilities and Network Diagnostics")

try:
    print("✓ DNS utilities and network diagnostics:")
    
    # 1. Network interface and IP utilities
    print("  1. Network interface and IP utilities:")
    
    # Test IP address utilities
    test_ips = [
        ('192.168.1.1', 'IPv4 private'),
        ('8.8.8.8', 'IPv4 public'),
        ('2001:db8::1', 'IPv6 documentation'),
        ('::1', 'IPv6 loopback'),
        ('fe80::1', 'IPv6 link-local')
    ]
    
    for ip_addr, description in test_ips:
        print(f"    IP address: {ip_addr} ({description})")
        
        # Determine IP version
        if ':' in ip_addr:
            # IPv6
            ipv6_binary = dns.ipv6.inet_aton(ip_addr)
            ipv6_text = dns.ipv6.inet_ntoa(ipv6_binary)
            
            print(f"      IPv6 validation: Valid")
            print(f"      Binary length: {len(ipv6_binary)} bytes")
            print(f"      Normalized: {ipv6_text}")
            assert (len(ipv6_binary), ipv6_text) == (16, ip_addr), (ip_addr, ipv6_text)
            
            # Check for special addresses
            if ip_addr.startswith('fe80:'):
                print(f"      Type: Link-local address")
            elif ip_addr == '::1':
                print(f"      Type: Loopback address")
            elif ip_addr.startswith('2001:db8:'):
                print(f"      Type: Documentation address")
            else:
                print(f"      Type: Global unicast address")
                
        else:
            # IPv4
            ipv4_binary = dns.ipv4.inet_aton(ip_addr)
            ipv4_text = dns.ipv4.inet_ntoa(ipv4_binary)
            
            print(f"      IPv4 validation: Valid")
            print(f"      Binary length: {len(ipv4_binary)} bytes")
            print(f"      Normalized: {ipv4_text}")
            assert (len(ipv4_binary), ipv4_text) == (4, ip_addr), (ip_addr, ipv4_text)
            
            # Check for private/special addresses
            ip_int = int.from_bytes(ipv4_binary, 'big')
            
            if (ip_int >= 0xC0A80000 and ip_int <= 0xC0A8FFFF) or \
               (ip_int >= 0xAC100000 and ip_int <= 0xAC1FFFFF) or \
               (ip_int >= 0x0A000000 and ip_int <= 0x0AFFFFFF):
                print(f"      Type: Private address")
            elif ip_int >= 0x7F000000 and ip_int <= 0x7FFFFFFF:
                print(f"      Type: Loopback address")
            else:
                print(f"      Type: Public address")
                
    # 2. DNS diagnostic queries
    print("  2. DNS diagnostic queries:")
    
    try:
        # Perform comprehensive DNS diagnostics
        diagnostic_domain = 'github.com'
        
        print(f"    DNS diagnostics for {diagnostic_domain}:")
        
        # Collect all record types
        record_types = ['A', 'AAAA', 'MX', 'NS', 'TXT', 'SOA']
        diagnostic_results = {}
        
        for record_type in record_types:
            try:
                start_time = time.time()
                result = dns.resolver.resolve(diagnostic_domain, record_type)
                query_time = (time.time() - start_time) * 1000
                
                records = [str(rdata) for rdata in result]
                diagnostic_results[record_type] = {
                    'records': records,
                    'count': len(records),
                    'ttl': result.ttl,
                    'query_time': query_time
                }
                
                print(f"      {record_type} records: {len(records)} found ({query_time:.1f}ms)")
                if records:
                    print(f"        Sample: {records[0][:50]}{'...' if len(records[0]) > 50 else ''}")
                    print(f"        TTL: {result.ttl}s")
                
            except dns.resolver.NoAnswer:
                diagnostic_results[record_type] = {'records': [], 'count': 0, 'error': 'NoAnswer'}
                print(f"      {record_type} records: No answer")
            except Exception as e:
                diagnostic_results[record_type] = {'records': [], 'count': 0, 'error': type(e).__name__}
                print(f"      {record_type} records: {type(e).__name__}")
            
            time.sleep(0.1)
        
        # Summary statistics
        total_records = sum(result.get('count', 0) for result in diagnostic_results.values())
        successful_queries = sum(1 for result in diagnostic_results.values() if result.get('count', 0) > 0)
        
        print(f"    Diagnostic summary:")
        print(f"      Total records found: {total_records}")
        print(f"      Successful query types: {successful_queries}/{len(record_types)}")
        
        # Check for IPv6 support
        has_ipv4 = diagnostic_results.get('A', {}).get('count', 0) > 0
        has_ipv6 = diagnostic_results.get('AAAA', {}).get('count', 0) > 0
        
        print(f"      IPv4 support: {'✓' if has_ipv4 else '✗'}")
        print(f"      IPv6 support: {'✓' if has_ipv6 else '✗'}")
        print(f"      Dual-stack: {'✓' if has_ipv4 and has_ipv6 else '✗'}")
        
    except Exception as e:
        print(f"    DNS diagnostics failed: {type(e).__name__}")
        raise
    
    # 3. DNS server performance testing
    print("  3. DNS server performance testing:")
    
    try:
        # Test different DNS servers
        dns_servers_perf = [
            ('Google Primary', '8.8.8.8'),
            ('Cloudflare Primary', '1.1.1.1'),
            ('Quad9 Primary', '9.9.9.9'),
            ('System Default', None)  # Use system default
        ]
        
        test_domain = 'example.com'
        performance_results = []
        
        for server_name, server_ip in dns_servers_perf:
            try:
                print(f"    Testing {server_name}:")
                
                # Create resolver
                if server_ip:
                    test_resolver = dns.resolver.Resolver()
                    test_resolver.nameservers = [server_ip]
                    test_resolver.timeout = 2.0
                    test_resolver.lifetime = 5.0
                else:
                    test_resolver = dns.resolver.get_default_resolver()
                
                # Perform multiple queries for average timing
                query_times = []
                
                for i in range(3):  # 3 test queries
                    try:
                        start_time = time.time()
                        result = test_resolver.resolve(test_domain, 'A')
                        end_time = time.time()
                        
                        query_time = (end_time - start_time) * 1000
                        query_times.append(query_time)
                        
                    except Exception:
                        continue
                    
                    time.sleep(0.1)
                
                if query_times:
                    avg_time = sum(query_times) / len(query_times)
                    min_time = min(query_times)
                    max_time = max(query_times)
                    
                    performance_results.append({
                        'name': server_name,
                        'server': server_ip or 'system',
                        'avg_time': avg_time,
                        'min_time': min_time,
                        'max_time': max_time,
                        'queries': len(query_times)
                    })
                    
                    print(f"      Server: {server_ip or 'system default'}")
                    print(f"      Average time: {avg_time:.1f}ms")
                    print(f"      Range: {min_time:.1f}ms - {max_time:.1f}ms")
                    print(f"      Successful queries: {len(query_times)}/3")
                else:
                    print(f"      All queries failed")
                
            except Exception as e:
                print(f"    {server_name} testing failed: {type(e).__name__}")
                raise
        
        # Find fastest server
        if performance_results:
            fastest = min(performance_results, key=lambda x: x['avg_time'])
            slowest = max(performance_results, key=lambda x: x['avg_time'])
            
            print(f"    Performance summary:")
            print(f"      Fastest: {fastest['name']} ({fastest['avg_time']:.1f}ms avg)")
            print(f"      Slowest: {slowest['name']} ({slowest['avg_time']:.1f}ms avg)")
            
            speed_diff = slowest['avg_time'] - fastest['avg_time']
            print(f"      Performance difference: {speed_diff:.1f}ms")
        
    except Exception as e:
        print(f"    DNS server performance testing failed: {type(e).__name__}")
        raise
    
    assert performance_results, "no DNS server (public or system default) answered an A query for example.com"
    
    # 4. Network connectivity diagnostics
    print("  4. Network connectivity diagnostics:")
    
    try:
        # Test connectivity to various services
        connectivity_tests = [
            ('Google DNS', '8.8.8.8', 53),
            ('Cloudflare DNS', '1.1.1.1', 53),
            ('HTTP (Google)', 'google.com', 80),
            ('HTTPS (GitHub)', 'github.com', 443),
        ]
        
        for service_name, host, port in connectivity_tests:
            try:
                print(f"    Testing {service_name}:")
                
                # First resolve the hostname if needed
                if not host.replace('.', '').isdigit() and ':' not in host:
                    # It's a hostname, resolve it first
                    try:
                        start_time = time.time()
                        dns_result = dns.resolver.resolve(host, 'A')
                        dns_time = (time.time() - start_time) * 1000
                        
                        target_ip = str(dns_result[0])
                        print(f"      DNS resolution: {host} -> {target_ip} ({dns_time:.1f}ms)")
                    except Exception as e:
                        print(f"      DNS resolution failed: {type(e).__name__}")
                        continue
                else:
                    target_ip = host
                    print(f"      Target IP: {target_ip}")
                
                # Test TCP connectivity
                try:
                    start_time = time.time()
                    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                    sock.settimeout(3.0)
                    
                    result = sock.connect_ex((target_ip, port))
                    connect_time = (time.time() - start_time) * 1000
                    
                    sock.close()
                    
                    if result == 0:
                        print(f"      TCP connection: Success ({connect_time:.1f}ms)")
                    else:
                        print(f"      TCP connection: Failed (error {result})")
                        
                except Exception as e:
                    print(f"      TCP connection failed: {type(e).__name__}")
                    raise
                
            except Exception as e:
                print(f"    {service_name} connectivity test failed: {type(e).__name__}")
                raise
            
            time.sleep(0.1)
        
    except Exception as e:
        print(f"    Network connectivity diagnostics failed: {type(e).__name__}")
        raise
    
    # 5. DNS cache and TTL analysis
    print("  5. DNS cache and TTL analysis:")
    
    try:
        # Analyze TTL behavior and caching
        ttl_test_domains = ['google.com', 'github.com', 'cloudflare.com']
        
        print(f"    TTL analysis:")
        
        for domain in ttl_test_domains[:2]:  # Test first 2 domains
            try:
                print(f"    Domain: {domain}")
                
                # Query multiple record types and analyze TTLs
                record_types_ttl = ['A', 'MX', 'NS']
                ttl_data = {}
                
                for record_type in record_types_ttl:
                    try:
                        result = dns.resolver.resolve(domain, record_type)
                        ttl_data[record_type] = result.ttl
                        
                        print(f"      {record_type} TTL: {result.ttl}s ({result.ttl/3600:.1f}h)")
                        
                    except (dns.resolver.NoAnswer, dns.resolver.NXDOMAIN):
                        print(f"      {record_type} TTL: No data")
                    except Exception as e:
                        print(f"      {record_type} TTL: Error ({type(e).__name__})")
                        raise
                    
                    time.sleep(0.1)
                
                # TTL analysis
                if ttl_data:
                    ttl_values = list(ttl_data.values())
                    min_ttl = min(ttl_values)
                    max_ttl = max(ttl_values)
                    avg_ttl = sum(ttl_values) / len(ttl_values)
                    
                    print(f"      TTL summary:")
                    print(f"        Min: {min_ttl}s ({min_ttl/3600:.1f}h)")
                    print(f"        Max: {max_ttl}s ({max_ttl/3600:.1f}h)")
                    print(f"        Average: {avg_ttl:.0f}s ({avg_ttl/3600:.1f}h)")
                    
                    # Cache effectiveness estimate
                    if min_ttl < 300:  # Less than 5 minutes
                        cache_effectiveness = "Low (frequent updates needed)"
                    elif min_ttl < 3600:  # Less than 1 hour
                        cache_effectiveness = "Medium (hourly updates)"
                    else:
                        cache_effectiveness = "High (long-term caching)"
                    
                    print(f"        Cache effectiveness: {cache_effectiveness}")
                
            except Exception as e:
                print(f"    TTL analysis for {domain} failed: {type(e).__name__}")
                raise
        
        # Overall DNS package capability summary
        print(f"    DNS package capabilities summary:")
        
        capabilities = [
            "✓ Basic DNS queries (A, AAAA, MX, NS, TXT, SOA)",
            "✓ Advanced resolution with custom servers",
            "✓ DNS zone management and record creation",
            "✓ DNSSEC support and validation",
            "✓ IP address utilities (IPv4/IPv6)",
            "✓ Network diagnostics and performance testing",
            "✓ DNS message construction and parsing",
            "✓ TTL analysis and caching behavior",
            "✓ Security analysis and amplification detection"
        ]
        
        for capability in capabilities:
            print(f"      {capability}")
        
        print(f"    Package modules used:")
        modules_used = [
            "dns.resolver", "dns.query", "dns.message", "dns.flags",
            "dns.zone", "dns.rdataset", "dns.rdata", "dns.rrset",
            "dns.dnssec", "dns.name", "dns.rdatatype", "dns.rdataclass",
            "dns.reversename", "dns.inet", "dns.ipv4", "dns.ipv6"
        ]
        
        for module in modules_used[:8]:  # Show first 8 modules
            print(f"      {module}")
        print(f"      ... and {len(modules_used) - 8} more modules")
        
    except Exception as e:
        print(f"    TTL analysis failed: {type(e).__name__}")
        raise
    
    print("✓ DNS utilities and network diagnostics completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dnspython example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")